In [2]:
import pandas as pd
df = pd.read_csv('/Users/zuhashaik/Research/SUN-lab/datasets/drugcombo-data/dose_level.csv')
df.head()

,PMID,NCTID,CombnID,Drug_Name,Dose_Level,Dose_Value,Dose_Unit,Frequency,Schedule,Comments
0,27853996,NCT01451632,4,seribantumab,-1,6,mg/kg,1,weekly,NaN
1,27853996,NCT01451632,4,cetuximab,-1,NaN,mg/m2,0,NaN,NaN
2,27853996,NCT01451632,4,cetuximab,-1,400,mg/m2,1,loading dose,NaN
3,27853996,NCT01451632,4,cetuximab,-1,200,mg/m2,1,weekly,NaN
4,27853996,NCT01451632,4,seribantumab,1,12,mg/kg,1,weekly,NaN


In [ ]:
import os
import json
import fitz
import faiss
import openai
from typing import List
from sentence_transformers import SentenceTransformer
import numpy as np
from openai import AzureOpenAI
    
with open('config.json', 'r') as config_file:
    config = json.load(config_file)

client = AzureOpenAI(
    api_key=config['api_key'],  
    api_version=config['api_version'],
    azure_endpoint=config['azure_endpoint']
)

def get_embedding(text):
    embedding = client.embeddings.create(
    input=text,
    model="text-embedding-2-small"
)
    return embedding.data[0].embedding

def extract_text_from_pdf(pdf_path: str) -> List[str]:
    doc = fitz.open(pdf_path)
    texts = [page.get_text() for page in doc]
    return texts


def chunk_text(texts: List[str], chunk_size: int = 500) -> List[str]:
    chunks = []
    for page in texts:
        for i in range(0, len(page), chunk_size):
            chunks.append(page[i:i + chunk_size])
    return chunks

def build_vector_db(chunks: List[str]):
    embeddings = [get_embedding(chunk) for chunk in chunks]
    dimension = len(embeddings[0])
    index = faiss.IndexFlatL2(dimension)
    index.add(np.array(embeddings).astype("float32"))
    return index, chunks

def retrieve_similar_chunks(index, chunks, query: str, k: int = 3):
    query_vec = get_embedding(query)
    D, I = index.search(np.array([query_vec]).astype("float32"), k)
    return [chunks[i] for i in I[0]]


def generate_answer(context_chunks: List[str], query: str) -> str:
    context = "\n\n".join(context_chunks)
    prompt = f"""You are an AI assistant extracting structured information from clinical trial documents.
Context: {context}
Question: What is the value of {query} in this document?
Answer:"""
    return generate_text(prompt)

def generate_text(prompt: str, max_tokens=500, temperature=0.3):
    response = client.chat.completions.create(
        model="gpt-35-turbo",
        messages=[{"role": "system", "content": "You are an AI assistant."},
                  {"role": "user", "content": prompt}],
        max_tokens=max_tokens,
        temperature=temperature
    )
    return response.choices[0].message.content

def rag_query(pdf_path: str, query: str):
    texts = extract_text_from_pdf(pdf_path)
    chunks = chunk_text(texts)
    index, all_chunks = build_vector_db(chunks)
    relevant_chunks = retrieve_similar_chunks(index, all_chunks, query)
    answer = generate_answer(relevant_chunks, query)
    return answer

/opt/anaconda3/envs/zuhas/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [51]:
generate_text('hey how are you?')

NotFoundError: Error code: 404 - {'error': {'code': 'DeploymentNotFound', 'message': 'The API deployment for this resource does not exist. If you created the deployment within the last 5 minutes, please wait a moment and try again.'}}

In [49]:
query = "Dose_Level"
pdf_path = "/Users/zuhashaik/Research/SUN-lab/datasets/drugcombo-data/pdfs_dataset/24770667.pdf"
pmid = pdf_path.split("/")[-1].split(".")[0]
print("PMID:", pmid)
df = df[df['PMID'] == int(pmid)]
df.head()


PMID: 24770667


,PMID,NCTID,CombnID,Drug_Name,Dose_Level,Dose_Value,Dose_Unit,Frequency,Schedule,Comments
546,24770667,NCT00633529,18,erlotinib,1,150,mg,1,days 1–21,on an empty stomach;fixed dose
547,24770667,NCT00633529,18,erlotinib,1,150,mg,1,days 1–21,on an empty stomach;fixed dose
548,24770667,NCT00633529,18,bevacizumab,1,15,mg/kg,1,on day 1 of each 3-week cycle,fixes dose
549,24770667,NCT00633529,18,imo-2055,1,0.08,mg/kg,1,"on days 1, 8, and 15 of each 3-week treatment ...",per week dose
550,24770667,NCT00633529,18,imo-2055,2,0.16,mg/kg,1,"on days 1, 8, and 15 of each 3-week treatment ...",per week dose


In [50]:
print("Query Answer:", rag_query(pdf_path, query='Drug_Name'))

NotFoundError: Error code: 404 - {'error': {'code': 'DeploymentNotFound', 'message': 'The API deployment for this resource does not exist. If you created the deployment within the last 5 minutes, please wait a moment and try again.'}}